import tensorflow as tf
print(tf.config.list_physical_devices('CPU'))

import tensorflow as tf
print(tf.config.threading.get_inter_op_parallelism_threads())
print(tf.config.threading.get_intra_op_parallelism_threads())

In [1]:
import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(2)

2025-04-02 11:44:01.155178: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
tf.debugging.set_log_device_placement(True)


In [3]:
import gc

tf.keras.backend.clear_session()  # Reset TensorFlow state
gc.collect()  # Free memory

0

In [4]:
import torch
from pathlib import Path
import numpy as np
import pandas as pd
import random
from glob import glob
import sklearn

from tqdm.autonotebook import tqdm
from sklearn.metrics import average_precision_score, roc_auc_score
from pathlib import Path

#set up plotting
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize']=[15,5] #for large visuals
%config InlineBackend.figure_format = 'retina'

# opensoundscape transfer learning tools
from opensoundscape.ml.shallow_classifier import MLPClassifier, quick_fit, fit_classifier_on_embeddings


/tmp/ipykernel_295652/3831688300.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [5]:
#specify site name
site = 'ankafobe_forest_A_KBF09'

site_labels = pd.read_csv(f'./data/{site}_5s.csv', index_col = [0,1,2])
site_labels.head()

Hypsipetes_madagascariensis  \
file                                               start_time end_time                                
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF09/K... 0          5                                 NaN   
                                                   5          10                                NaN   
                                                   10         15                                NaN   
                                                   15         20                                NaN   
                                                   20         25                                NaN   

                                                                        Copsychus_albospecularis  \
file                                               start_time end_time                             
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF09/K... 0          5                              NaN   
                                                   5          10                             NaN   
                                                   10         15                             NaN   
                                                   15         20                             NaN   
                                                   20         25                             NaN   

                                                                        Coracopsis_nigra  \
file                                               start_time end_time                     
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF09/K... 0          5                      NaN   
                                                   5          10                     NaN   
                                                   10         15                     NaN   
                                                   15         20                     NaN   
                                                   20         25                     NaN   

                                                                        Dicrurus_forficatus  \
file                                               start_time end_time                        
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF09/K... 0          5                         NaN   
                                                   5          10                        NaN   
                                                   10         15                        NaN   
                                                   15         20                        NaN   
                                                   20         25                        NaN   

                                                                        Coua_caerulea  \
file                                               start_time end_time                  
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF09/K... 0          5                   NaN   
                                                   5          10                  NaN   
                                                   10         15                  NaN   
                                                   15         20                  NaN   
                                                   20         25                  NaN   

                                                                        Zosterops_maderaspatanus  \
file                                               start_time end_time                             
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF09/K... 0          5                              NaN   
                                                   5          10                             NaN   
                                                   10         15                             NaN   
                                                   15         20                             NaN   
                                                   20         25                             NaN   

             

In [6]:
# pick classes to train the model on. These should occur in the annotated data
class_list = ['Hypsipetes_madagascariensis','Copsychus_albospecularis','Coracopsis_nigra','Dicrurus_forficatus','Coua_caerulea','Zosterops_maderaspatanus','Eurystomus_glaucurus','Agapornis_canus','Saxicola_torquatus','Cyanolanius_madagascarinus','Leptopterus_chabert','Nesoenas_picturatus','Coua_reynaudii','Ceblepyris_cinereus','Neodrepanis_coruscans','Philepitta_castanea','Eulemur_sp','Coua_cristata','Treron_australis']


In [7]:
from bioacoustics_model_zoo import Perch

In [8]:
#specify the name of the file you want to load
filename = 'Shallow_classifier_perch_resample'


In [9]:
perch_model = Perch.load(f'./Perch/{filename}.model')

/home/sholmes3/Linux/miniconda3/envs/rewilding_tensorflow_wsl/lib/python3.10/site-packages/bioacoustics_model_zoo/tensorflow_wrapper.py:224: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experiment

resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
resource_RetVal:

2025-04-02 11:44:18.542612: I tensorflow/core/common_runtime/placer.cc:162] resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.542940: I tensorflow/core/common_runtime/placer.cc:162] VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.564476: I tensorflow/core/common_runtime/placer.cc:162] resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.564517: I tensorflow/core/common_runtime/placer.cc:162] VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.568058: I tensorflow/core/common_runtime/placer.cc:162] resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.568154: I tensorflow/core/common_runtime/placer.cc:162] VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.569545: I tensorflow/core/common_runtime/placer.cc:162] resource_RetVal: (_Retval): /j

Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op VarHandleOp in device /job:localhost/replica:0/task:0/device:CPU:0
Ex

2025-04-02 11:44:18.746606: I tensorflow/core/common_runtime/placer.cc:162] resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.746651: I tensorflow/core/common_runtime/placer.cc:162] VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.748750: I tensorflow/core/common_runtime/placer.cc:162] resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.748785: I tensorflow/core/common_runtime/placer.cc:162] VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.752670: I tensorflow/core/common_runtime/placer.cc:162] resource_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.752733: I tensorflow/core/common_runtime/placer.cc:162] VarHandleOp: (VarHandleOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:18.755893: I tensorflow/core/common_runtime/placer.cc:162] resource_RetVal: (_Retval): /j

...

2025-04-02 11:44:20.272323: I tensorflow/core/common_runtime/placer.cc:162] resource: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.272363: I tensorflow/core/common_runtime/placer.cc:162] value: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.272372: I tensorflow/core/common_runtime/placer.cc:162] AssignVariableOp: (AssignVariableOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.327721: I tensorflow/core/common_runtime/placer.cc:162] resource: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.327763: I tensorflow/core/common_runtime/placer.cc:162] value: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.327771: I tensorflow/core/common_runtime/placer.cc:162] AssignVariableOp: (AssignVariableOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.346736: I tensorflow/core/common_runtime/placer.cc:162] resource: (_Arg): /job:localhost/replica:0/task:0/device

Executing op Identity in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AssignVariableOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op NoOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Identity in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AssignVariableOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op NoOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Identity in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AssignVariableOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op NoOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Identity in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AssignVariableOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op NoOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Identity in device /job:localhost/replica:0/tas

2025-04-02 11:44:20.520377: I tensorflow/core/common_runtime/placer.cc:162] resource: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.520417: I tensorflow/core/common_runtime/placer.cc:162] value: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.520426: I tensorflow/core/common_runtime/placer.cc:162] AssignVariableOp: (AssignVariableOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.541604: I tensorflow/core/common_runtime/placer.cc:162] resource: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.541642: I tensorflow/core/common_runtime/placer.cc:162] value: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.541661: I tensorflow/core/common_runtime/placer.cc:162] AssignVariableOp: (AssignVariableOp): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-02 11:44:20.564996: I tensorflow/core/common_runtime/placer.cc:162] resource: (_Arg): /job:localhost/replica:0/task:0/device

Executing op AssignVariableOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op NoOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Identity in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AssignVariableOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op NoOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Identity in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AssignVariableOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op NoOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Identity in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AssignVariableOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op NoOp in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op Identity in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op AssignVariableOp in device /job:localhost/repli

/home/sholmes3/Linux/miniconda3/envs/rewilding_tensorflow_wsl/lib/python3.10/site-packages/opensoundscape/ml/cnn.py:621: UserWarning: 
                    This architecture is not listed in opensoundscape.ml.cnn_architectures.ARCH_DICT.
                    It will not be available for loading after saving the model with .save() (unless using pickle=True). 
                    To make it re-loadable, define a function that generates the architecture from arguments: (n_classes, n_channels) 
                    then use opensoundscape.ml.cnn_architectures.register_architecture() to register the generating function.

                    The function can also set the returned object's .constructor_name to the registered string key in ARCH_DICT
                    to avoid this warning and ensure it is reloaded correctly by opensoundscape.ml.load_model().

                    See opensoundscape.ml.cnn_architectures module for examples of constructor functions
                    
  warnings.

In [10]:
perch_model.use_custom_classifier

True

In [ ]:
#Test to make sure WSL can access files in hard drive

import opensoundscape

# Import Audio class from OpenSoundscape
from opensoundscape import Audio, audio

# load an audio file from a file
# can be any file path to an audio file on your computer
path = "/mnt/d/Sound_files/Marojejy/ANT_A/ANT_A-ANT02/ANT_A-ANT02_20230407_140000.WAV"
audio_object = Audio.from_file(path)

# returning the audio object from a cell will display a player widget
audio_object

In [ ]:
# make predictions by passing the embeddings through the classifier
preds = perch_model.predict(site_labels,batch_size=1,num_workers=0)

preds

  0%|          | 0/12996 [00:00<?, ?it/s]

Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
input: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
_EagerConst: (_EagerConst): /job:localhost/replica:0/task:0/device:CPU:0
output_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0


2025-04-01 16:35:25.812007: I tensorflow/core/common_runtime/placer.cc:162] input: (_Arg): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-01 16:35:25.812075: I tensorflow/core/common_runtime/placer.cc:162] _EagerConst: (_EagerConst): /job:localhost/replica:0/task:0/device:CPU:0
2025-04-01 16:35:25.812084: I tensorflow/core/common_runtime/placer.cc:162] output_RetVal: (_Retval): /job:localhost/replica:0/task:0/device:CPU:0


Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0


I0000 00:00:1743518126.097440     933 service.cc:152] XLA service 0x3b12e160 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1743518126.097484     933 service.cc:160]   StreamExecutor device (0): Host, Default Version
2025-04-01 16:35:26.417644: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-04-01 16:35:26.490631: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator jax2tf_infer_fn_/assert_equal_1/Assert/AssertGuard/Assert
I0000 00:00:1743518128.054802     933 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/repli

/home/sholmes3/Linux/miniconda3/envs/rewilding_tensorflow_wsl/lib/python3.10/site-packages/opensoundscape/audio.py:1725: UserWarning: Audio object is shorter than requested duration: 4.0 sec instead of 5 sec
  warnings.warn(error_msg)


Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op __inference_restored_function_body_20537 in device /job:localhost/replica:0/task:0/device:CPU:0
Executing op _EagerConst in device /job:localhost/repli

Hypsipetes_madagascariensis  \
file                                               start_time end_time                                
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF02/K... 0          5                           -4.888137   
                                                   5          10                          -5.177096   
                                                   10         15                          -5.246050   
                                                   15         20                          -5.519248   
                                                   20         25                          -5.270496   
...                                                                                             ...   
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF02/K... 35         40                          -4.621851   
                                                   40         45                          -6.067334   
                                                   45         50                          -5.087271   
                                                   50         55                          -5.702435   
                                                   55         60                          -5.246199   

                                                                        Copsychus_albospecularis  \
file                                               start_time end_time                             
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF02/K... 0          5                        -5.043074   
                                                   5          10                       -3.551163   
                                                   10         15                       -3.279230   
                                                   15         20                       -3.938374   
                                                   20         25                       -3.870216   
...                                                                                          ...   
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF02/K... 35         40                       -3.949428   
                                                   40         45                       -4.021196   
                                                   45         50                       -3.381906   
                                                   50         55                       -4.460399   
                                                   55         60                       -4.166567   

                                                                        Coracopsis_nigra  \
file                                               start_time end_time                     
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF02/K... 0          5                -3.320585   
                                                   5          10               -5.202821   
                                                   10         15               -4.797756   
                                                   15         20               -4.358291   
                                                   20         25               -5.073167   
...                                                                                  ...   
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF02/K... 35         40               -4.555232   
                                                   40         45               -4.722489   
                                                   45         50               -4.341614   
                                                   50         55               -4.974251   
                                                   55         60               -3.887368   

                                                                        Dicrurus_forficatus  \
file                                               start_time end_time                        
/mnt/d/Sound_files/Ankafobe/KBF_A/KBF_A-KBF0

In [ ]:
preds.to_csv(f'./results/{filename}_{site}_predictions.csv')